# Householder QR


In [6]:
import time
import sys
from pathlib import Path
import numpy as np
from scipy.sparse import csr_matrix, issparse

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "common.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start the notebook inside Numerical_Linear_Algebra_Algorithms")
sys.path.insert(0, str(ROOT))
from common import loader, data_path, lu_solve, relative_residual
from Orthogonalization.householder_qr import householder_qr, apply_q


def timed(fn, *args):
    start = time.perf_counter()
    result = fn(*args)
    return result, time.perf_counter() - start


## 3×3


In [7]:
A = np.array([[4., 1., 0.], [1., 3., 1.], [0., 1., 2.]])
x_true = np.ones(3)
b = A @ x_true
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))
_ = householder_qr(A)
result, seconds = timed(householder_qr, A)
print("Seconds:", seconds)
V, R = result
print("Factorization probe error:", float(np.linalg.norm(A @ Z - apply_q(V, R @ Z)) / np.linalg.norm(A @ Z)))
Y = rng.normal(size=(A.shape[0], 3))
print("Orthogonality probe error:", float(np.linalg.norm(apply_q(V, apply_q(V, Y), transpose=True)-Y) / np.linalg.norm(Y)))
assert float(np.linalg.norm(A @ Z - apply_q(V, R @ Z)) / np.linalg.norm(A @ Z)) < 1e-12


Seconds: 8.333998266607523e-06
Factorization probe error: 6.760058868449193e-16
Orthogonality probe error: 7.320662154861416e-16


## SuiteSparse

Change MATRIX to an existing file. bcsstk09 is 1,083×1,083; bcsstk17 is 10,974×10,974; west1505 is nonsymmetric and requires pivoting. No 10^5 matrix exists in this repo. Dense factorization costs O(n³); run the optional 10^4 case deliberately.


In [8]:
MATRIX = "bcsstk09"
MAT_PATH = None
A = loader("suitesparse", path=data_path(MATRIX, MAT_PATH), dense=True)
print("Matrix:", MATRIX, "shape:", A.shape)
rng = np.random.default_rng(42)
Z = rng.normal(size=(A.shape[1], 3))


Matrix: bcsstk09 shape: (1083, 1083)


In [9]:
result, seconds = timed(householder_qr, A)
print("Seconds:", seconds)
V, R = result
print("Factorization probe error:", float(np.linalg.norm(A @ Z - apply_q(V, R @ Z)) / np.linalg.norm(A @ Z)))
Y = rng.normal(size=(A.shape[0], 3))
print("Orthogonality probe error:", float(np.linalg.norm(apply_q(V, apply_q(V, Y), transpose=True)-Y) / np.linalg.norm(Y)))


Seconds: 0.827293207985349
Factorization probe error: 1.4452524304501494e-15
Orthogonality probe error: 1.9652788506677113e-15
